# 어텐션을 통한 역전파: 단계별 (상세 설명판) - 어텐션 역전파의 수학

- Tutorial ID: `ull-3`
- Tutorial: 어텐션을 통한 역전파: 단계별
- Section ID: `ull-3-1`
- Section: 어텐션 역전파의 수학

> 이 노트북은 같은 주제를 다루는 원본 노트북을 **처음 공부하는 사람** 기준으로 다시 쓴 버전입니다.
> 수식을 코드로 옮기기 전에, 그 수식이 "무엇을 하려는 것인지"를 먼저 말로 풀어서 설명하고,
> 작은 손으로 셀 수 있는 예제로 직접 확인한 다음, 마지막에 실제 어텐션 계산에 적용합니다.


## 이 노트북에서 배우는 것

순전파(forward pass)는 "입력 → 예측"으로 정보가 앞으로 흘러가는 과정입니다.
역전파(backward pass)는 그 반대로, "손실(loss, 오차) → 입력 방향"으로
**"여기를 이만큼 바꾸면 손실이 이만큼 줄어든다"** 라는 신호를 거꾸로 전달하는 과정입니다.

비유를 하나 들면: 시험 점수가 낮게 나왔을 때, 답안지를 맨 뒤에서부터(최종 점수에서부터) 거슬러 올라가며
"몇 번 문제에서 어떤 실수를 했길래 점수가 깎였는지"를 되짚어보는 것과 비슷합니다. 신경망 학습에서는
이 "되짚어보기"를 사람이 손으로 하지 않고, **체인룰(연쇄법칙)** 이라는 미분 규칙을 이용해 자동으로 계산합니다.
이 노트북에서는 그 체인룰을 어텐션(attention) 연산 전체에 직접 적용해 봅니다.

**학습 목표**

1. Q/K/V가 어떤 shape으로 만들어지고, 어텐션 score → 확률(softmax) → 출력으로 이어지는 과정을 직접 추적합니다.
2. 손실(loss)에서부터 거꾸로, 체인룰을 이용해 모든 가중치(`W_Q, W_K, W_V, W_O`)에 대한 그래디언트를 손으로 유도합니다.
3. 그중 가장 어려운 부분인 **softmax의 역전파(야코비안)** 를 작은 예제로 충분히 연습한 뒤 적용합니다.
4. **수치 미분(finite difference)** 으로 우리가 손으로 유도한 그래디언트가 맞는지 직접 검증합니다.

**읽는 순서 (이 노트북의 구성)**

- 0단계 — 워밍업: 체인룰과 "행렬곱의 그래디언트 규칙"을 아주 작은 예제로 먼저 연습합니다.
- 1단계 — 어텐션이 하는 일을 다시 한 번 말로 정리합니다 (Q, K, V가 각각 무엇을 의미하는지).
- 2단계 — 순전파: 작은 숫자 예제로 Q, K, V, score, softmax, 출력까지 한 줄씩 계산합니다.
- 3단계 — 역전파: 손실에서부터 거꾸로, 0단계에서 연습한 규칙들을 그대로 재사용해 모든 그래디언트를 구합니다.
- 4단계 — 수치 미분으로 전체 결과를 검증합니다.
- 보너스 — (선택) PyTorch의 자동미분(autograd)이 같은 답을 내는지 비교합니다.

**미리 알아두면 좋은 것**: 고등학교 수준의 미분(연쇄법칙)과 행렬곱이 무엇인지 정도만 알면 충분합니다.
나머지는 이 노트북 안에서 그때그때 설명합니다. 새로운 용어가 나오면 바로 위/아래에 말로 풀어서 설명을 붙였으니,
모르는 용어가 나와도 당황하지 말고 천천히 따라오시면 됩니다.


## 0단계 — 워밍업 ①: 체인룰(연쇄법칙) 감 잡기

본격적으로 어텐션을 다루기 전에, 아주 단순한 숫자 함수로 체인룰을 다시 한 번 확인하고 갑니다.
나중에 어텐션의 역전파를 볼 때도 결국 이 워밍업과 똑같은 논리를 반복해서 적용할 뿐입니다.

두 개의 함수를 이어 붙인다고 생각해 봅시다.

$$ u = f(x) = x^2 \qquad y = g(u) = 3u + 5 $$

이 둘을 합친 함수를 $h(x) = g(f(x))$ 라고 하면, $h$가 $x$에 따라 얼마나 변하는지($dh/dx$)는
중간 변수 $u$를 거쳐서 다음처럼 "곱"으로 연결됩니다. 이것이 바로 체인룰입니다.

$$ \frac{dh}{dx} = \frac{dg}{du} \cdot \frac{df}{dx} $$

말로 풀면: "$x$가 조금 바뀌면 $u$가 얼마나 바뀌는지" $\times$ "$u$가 조금 바뀌면 $y$가 얼마나 바뀌는지"를
곱하면, "$x$가 조금 바뀌면 $y$가 최종적으로 얼마나 바뀌는지"를 알 수 있다는 뜻입니다.
신경망의 역전파는 이 곱셈을 **층(layer)이 아주 많은 경우**로 그대로 확장한 것뿐입니다.

여기서는 $dg/du = 3$ (상수이므로 항상 3), $df/dx = 2x$ 이므로 $dh/dx = 3 \cdot 2x = 6x$ 입니다.
아래 코드에서 이 값을 직접 계산하고, **수치 미분(finite difference)** 이라는 방법으로도 같은 값이 나오는지 확인합니다.

> **수치 미분이란?** 미분의 정의 $\frac{df}{dx} = \lim_{\epsilon \to 0} \frac{f(x+\epsilon)-f(x-\epsilon)}{2\epsilon}$ 을
> 그대로 코드로 옮긴 것입니다. $\epsilon$(eps)을 아주 작은 값(예: 0.00001)으로 두고
> $x$를 살짝 늘렸을 때와 살짝 줄였을 때의 함숫값 차이를 보면, 우리가 손으로 유도한 미분 공식이
> 맞는지를 "정답지 없이도" 검증할 수 있습니다. 이 노트북 전체에서 이 방법을 계속 재사용합니다.


In [ ]:
# 워밍업 ①: 체인룰을 아주 단순한 숫자 함수로 확인하기
import numpy as np

def f(x):
    return x ** 2          # u = f(x) = x^2

def g(u):
    return 3 * u + 5       # y = g(u) = 3u + 5

def h(x):
    return g(f(x))         # h(x) = g(f(x)) : f와 g를 이어붙인 합성함수

x0 = 2.0

# 손으로 유도한(해석적) 미분: dh/dx = dg/du * df/dx = 3 * (2x)
dg_du = 3.0
df_dx = 2 * x0
analytic_grad = dg_du * df_dx

# 수치 미분: (h(x+eps) - h(x-eps)) / (2*eps)
eps = 1e-5
numerical_grad = (h(x0 + eps) - h(x0 - eps)) / (2 * eps)

print(f"x0 = {x0}")
print(f"체인룰로 손으로 구한 dh/dx     : {analytic_grad:.8f}")
print(f"수치 미분으로 확인한 dh/dx     : {numerical_grad:.8f}")
print(f"차이                          : {abs(analytic_grad - numerical_grad):.2e}  (0에 아주 가까워야 정상)")

## 0단계 — 워밍업 ②: 행렬곱의 그래디언트 "황금 규칙" 두 가지

어텐션 내부의 거의 모든 연산은 결국 **행렬곱**입니다 ($Q=XW_Q$, $S=QK^\top$, $O=AV$, $Y=OW_O$ 등).
그래서 "행렬곱을 역전파할 때 그래디언트가 어떻게 흐르는지"를 한 번만 제대로 이해하면,
이 노트북에 나오는 거의 모든 미분을 직접 손으로 계산할 수 있습니다. 이것을 미리 연습해 둡니다.

행렬 $A$ ($m\times k$ 크기)와 $B$ ($k\times n$ 크기)를 곱해 $Y = AB$ ($m\times n$ 크기)를 만들었다고 합시다.
그리고 최종 손실 $L$에 대해 $Y$의 그래디언트 $\frac{\partial L}{\partial Y}$ (줄여서 $dY$, $Y$와 같은 모양)를
이미 알고 있다고 합시다. (역전파는 항상 "출력 쪽 그래디언트를 먼저 알고, 입력 쪽 그래디언트를 구하는" 순서로 진행됩니다.)

이때 다음 두 규칙이 성립합니다 (행렬 미분의 표준 결과이며, 코드 셀에서 직접 수치적으로 확인합니다).

$$ \textbf{규칙 1: } \quad \frac{\partial L}{\partial A} = dY \, B^\top $$
$$ \textbf{규칙 2: } \quad \frac{\partial L}{\partial B} = A^\top \, dY $$

**외우는 방법(직관)**: $Y=AB$ 라는 식에서 구하고 싶은 변수를 식의 왼쪽으로 옮긴다고 생각하면 됩니다.
- $A$에 대한 그래디언트를 구할 땐 $B$를 "뒤집어서(transpose) $dY$의 오른쪽에" 곱합니다 → $dY B^\top$
- $B$에 대한 그래디언트를 구할 땐 $A$를 "뒤집어서(transpose) $dY$의 왼쪽에" 곱합니다 → $A^\top dY$

그리고 shape도 항상 맞습니다: $dA$는 $A$와 같은 $(m,k)$, $dB$는 $B$와 같은 $(k,n)$ 모양이 되어야 하는데,
$dY(m,n) \cdot B^\top(n,k) = (m,k)$, $A^\top(k,m) \cdot dY(m,n) = (k,n)$ 으로 정확히 맞아떨어집니다.
**shape이 맞는지 확인하는 습관**은 행렬 미분에서 실수를 줄이는 아주 좋은 방법입니다.

이 두 규칙을 이 노트북 전체에서 "Q에 대한 그래디언트", "X에 대한 그래디언트" 등을 구할 때마다 반복해서 사용할 것입니다.


In [ ]:
# 워밍업 ②: 행렬곱의 그래디언트 황금 규칙 두 가지를 작은 행렬로 직접 확인하기
rng = np.random.default_rng(0)

A_demo = rng.normal(size=(3, 2))   # shape (m=3, k=2)
B_demo = rng.normal(size=(2, 4))   # shape (k=2, n=4)

def toy_loss(A_, B_):
    # 워밍업 1과 마찬가지로 "출력의 제곱합을 절반으로" 라는 단순한 손실을 사용합니다.
    # (실제 어텐션 예제에서도 같은 형태의 손실을 사용할 것이라, 미리 같은 패턴으로 연습합니다.)
    Y_ = A_ @ B_
    return np.sum(Y_ ** 2) / 2

Y_demo = A_demo @ B_demo
dL_dY_demo = Y_demo.copy()          # L = sum(Y^2)/2 이므로 dL/dY = Y (워밍업 1과 동일한 패턴)

# 황금 규칙 적용
dL_dA_demo = dL_dY_demo @ B_demo.T   # 규칙 1
dL_dB_demo = A_demo.T @ dL_dY_demo   # 규칙 2

print("dA shape:", dL_dA_demo.shape, " (A와 동일해야 함:", A_demo.shape, ")")
print("dB shape:", dL_dB_demo.shape, " (B와 동일해야 함:", B_demo.shape, ")")

# 수치 미분으로 각 규칙을 원소 하나씩 검증
eps = 1e-6

i, j = 1, 0
Ap = A_demo.copy(); Ap[i, j] += eps
Am = A_demo.copy(); Am[i, j] -= eps
num_dA = (toy_loss(Ap, B_demo) - toy_loss(Am, B_demo)) / (2 * eps)
print(f"\n규칙 1 검증 (dA[{i},{j}]):  해석적 {dL_dA_demo[i,j]:.6f}  vs  수치적 {num_dA:.6f}")

i, j = 1, 2
Bp = B_demo.copy(); Bp[i, j] += eps
Bm = B_demo.copy(); Bm[i, j] -= eps
num_dB = (toy_loss(A_demo, Bp) - toy_loss(A_demo, Bm)) / (2 * eps)
print(f"규칙 2 검증 (dB[{i},{j}]):  해석적 {dL_dB_demo[i,j]:.6f}  vs  수치적 {num_dB:.6f}")
print("\n두 값이 거의 같다면, 이 두 황금 규칙을 믿고 앞으로 계속 사용해도 좋습니다.")

## 1단계 — 어텐션이 하는 일을 다시 정리하기

워밍업이 끝났으니, 이제 진짜 주제인 **어텐션(attention)** 으로 들어갑니다.
역전파를 이해하려면 먼저 순전파에서 각 글자(Q, K, V, S, A, O, Y)가 "무엇을 의미하는지" 감이 있어야 합니다.

**비유: 도서관에서 책 찾기**

- **Query(질문, $Q$)**: "나는 지금 이런 정보가 필요해" 라는 질문입니다. 문장의 각 토큰(단어)이 자기 입장에서
  "다른 토큰들로부터 어떤 정보를 가져오고 싶은지"를 표현한 벡터입니다.
- **Key(색인표, $K$)**: 각 토큰이 "나는 이런 정보를 갖고 있어" 라고 내거는 색인표(꼬리표)입니다.
- **Value(실제 내용, $V$)**: 그 토큰이 실제로 갖고 있는 내용물입니다.

어텐션은 **내 질문($Q$)과 다른 토큰들의 색인표($K$)를 하나씩 비교(내적)** 해서 "누구의 정보가
지금 나에게 더 필요한지" 점수를 매기고, 그 점수를 0~1 사이 확률(softmax)로 바꾼 다음,
**그 확률을 가중치로 사용해 각 토큰의 실제 내용물($V$)을 섞어서** 가져옵니다.
즉, 매칭 점수가 높을수록 그 토큰의 내용을 더 많이 가져오는 "가중 평균"인 셈입니다.

**왜 미래 토큰을 가려야 할까? (Causal mask, 인과적 마스크)**

GPT 계열 언어모델은 문장을 왼쪽에서 오른쪽으로 한 토큰씩 만들어 냅니다 (예측 → 다음 단어 예측 → ...).
그런데 학습 중에는 문장 전체가 이미 주어져 있기 때문에, 만약 아무 제약이 없다면 모델이 "미래의 정답"을
미리 들여다보고 베껴서 맞추는 부정행위를 할 수 있습니다. 이를 막기 위해, **각 토큰은 자기 자신과 그 이전
토큰까지만 볼 수 있고, 미래 토큰은 점수를 $-\infty$로 만들어 강제로 가립니다.** softmax를 거치면
$-\infty$였던 점수는 확률 0이 되어, 미래 토큰의 내용은 전혀 섞이지 않게 됩니다.

**이 노트북에서 사용할 예제 문장**

아래 코드에서는 `"나는 / 사과를 / 좋아한다"` 라는 3개 토큰짜리 문장을 예로 사용합니다.
(실제로는 학습된 임베딩을 사용하지만, 여기서는 shape과 흐름을 추적하기 쉽도록 무작위(random) 숫자를
"토큰 임베딩이라고 치고" 사용합니다 — 숫자 자체의 의미보다 **shape이 어떻게 바뀌는지, 정보가 어느 방향으로
흐르는지**에 집중하세요.) 인과적 마스크 때문에:

- "나는" (1번째 토큰)은 자기 자신만 볼 수 있습니다.
- "사과를" (2번째 토큰)은 "나는"과 "사과를"까지 볼 수 있습니다.
- "좋아한다" (3번째 토큰)는 세 토큰을 모두 볼 수 있습니다.

이 구조가 실제로 어텐션 가중치 행렬에 어떻게 나타나는지 잠시 뒤 코드 출력에서 직접 확인할 것입니다.

**전체 계산 흐름 (앞으로 계속 참조할 지도)**

```
X (입력, 토큰 임베딩)
 ├─ W_Q → Q
 ├─ W_K → K
 └─ W_V → V

Q, K           →  S = Q K^T / √d_head        (점수, score)
S + causal mask →  S_masked                   (미래 토큰 가리기)
S_masked       →  A = softmax(S_masked)        (확률로 변환)
A, V           →  O = A V                      (값들을 확률로 가중 평균)
O              →  Y = O W_O                    (다시 d_model 차원으로)
Y              →  L = sum(Y^2) / 2             (이번 노트북에서 쓸 단순한 손실)
```

이 지도에서 **오른쪽에서 왼쪽으로** 거슬러 올라가는 것이 바로 3단계에서 할 역전파입니다.


In [ ]:
# 실습에 사용할 예제와 하이퍼파라미터(크기) 설정
print("=" * 62)
print("어텐션 역전파: 단계별 추적 (상세 설명판)")
print("=" * 62)

def softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))  # 큰 값 빼서 overflow 방지 (값 자체는 안 변함)
    return e / e.sum(axis=axis, keepdims=True)

np.random.seed(42)  # 같은 무작위 값이 매번 나오도록 고정 (재현성)

# --- 하이퍼파라미터 (작은 숫자로 설정해서 손으로도 따라갈 수 있게 합니다) ---
tokens = ["나는", "사과를", "좋아한다"]   # 예제 문장 (3개 토큰)
seq_len = len(tokens)   # 문장의 토큰 개수 = 3
d_model = 4             # 토큰 하나를 표현하는 임베딩 벡터의 차원
d_head  = 2             # 어텐션 내부에서 Q/K/V가 투영되는(projected) 더 작은 차원

# X: 입력 토큰 임베딩. 실제로는 학습된 임베딩이지만, 여기서는 "토큰 임베딩이라고 치고" 무작위 숫자를 사용합니다.
X = np.random.randn(seq_len, d_model) * 0.5

# W_Q, W_K, W_V: 각각 X를 Q공간/K공간/V공간으로 보내는 가중치 행렬 (학습되는 파라미터)
W_Q = np.random.randn(d_model, d_head) * 0.3
W_K = np.random.randn(d_model, d_head) * 0.3
W_V = np.random.randn(d_model, d_head) * 0.3
# W_O: 어텐션 결과(d_head 차원)를 다시 원래 d_model 차원으로 되돌리는 출력 가중치
W_O = np.random.randn(d_head, d_model) * 0.3

print(f"\n토큰: {tokens}")
print(f"seq_len={seq_len}, d_model={d_model}, d_head={d_head}")
print(f"\nX (입력 임베딩, shape {X.shape}):")
for tok, row in zip(tokens, X):
    print(f"  {tok:6s}: {row}")

## 2단계 — 순전파(Forward Pass): 한 줄씩 따라가기

이제 1단계에서 그린 "지도"를 실제 숫자로 한 단계씩 따라가 봅니다. 코드를 실행하기 전에,
각 줄이 *왜* 그렇게 계산되는지 먼저 짚어보겠습니다.

**① $Q = XW_Q$, $K = XW_K$, $V = XW_V$**
같은 입력 $X$를 세 개의 다른 가중치 행렬에 곱해서, 같은 토큰이라도 "질문 역할", "색인표 역할",
"내용물 역할"을 할 때 서로 다른 벡터가 되도록 만듭니다. shape은 $X$가 $(\text{seq\_len}, d_{model})$,
$W_Q$가 $(d_{model}, d_{head})$이므로 $Q$는 $(\text{seq\_len}, d_{head})$가 됩니다. ($K$, $V$도 동일)

**② $S = QK^\top / \sqrt{d_{head}}$ (어텐션 점수)**
$Q$의 각 행(토큰)과 $K$의 각 행(토큰)을 내적해서, "이 토큰이 저 토큰을 얼마나 필요로 하는지" 점수를 매깁니다.
$\sqrt{d_{head}}$로 나누는 이유: 벡터의 차원이 커질수록 내적 값의 분산(들쭉날쭉한 정도)이 커지는 경향이 있는데,
점수가 너무 크면 softmax가 한쪽으로 치우쳐 거의 0과 1로만 쏠리게 되고, 그러면 학습 중 그래디언트가
거의 0이 되어버립니다(softmax 포화). 이를 막기 위해 차원 수의 제곱근으로 점수를 미리 눌러줍니다.

**③ 인과적 마스크 (Causal mask)**
$i$번째 토큰이 $j$번째 토큰(미래, $j>i$)을 보지 못하게, 그 위치의 점수에 매우 작은 값($-10^9$)을 더합니다.
($-\infty$를 직접 쓰면 계산상 `nan`이 생길 위험이 있어, 충분히 작은 큰 음수로 흉내냅니다.)

**④ $A = \text{softmax}(S_{masked})$**
각 행(토큰)별로 점수를 0~1 사이 확률로 바꿔주어, 한 행의 합이 항상 1이 되도록 만듭니다.
마스크로 $-10^9$가 된 위치는 $e^{-10^9} \approx 0$ 이라서 softmax를 통과하면 확률 0이 됩니다 — 즉, 미래
토큰의 정보는 다음 단계에서 전혀 섞이지 않습니다.

**⑤ $O = AV$**
각 토큰의 출력은 "다른 토큰들의 내용물($V$)을, 방금 구한 확률($A$)을 가중치로 섞은 평균"입니다.

**⑥ $Y = OW_O$**
$d_{head}$ 차원이었던 어텐션 결과를 다시 $d_{model}$ 차원으로 되돌려, 다음 레이어(또는 출력)와
차원을 맞춥니다.

**⑦ 손실(Loss): $L = \frac{1}{2}\sum Y^2$**
실제 언어모델은 보통 cross-entropy 손실을 쓰지만, 여기서는 미분이 아주 단순한
$L=\frac12\sum Y^2$ (제곱합의 절반)을 사용합니다. 워밍업 1에서 본 $x^2/2$의 미분이 $x$였던 것과
똑같은 이유로, $\partial L/\partial Y = Y$ 라는 깔끔한 출발점을 얻을 수 있습니다. **역전파의 메커니즘
자체는 손실 함수의 종류와 무관하게 동일**하며, 손실 함수가 바뀌면 오직 이 맨 처음 출발점만 바뀐다는
점을 기억해 두면 좋습니다.

자, 이제 코드로 옮겨봅니다. 각 줄 출력에서 shape뿐 아니라 **어텐션 가중치 행렬 $A$의 실제 숫자**도
함께 확인해서, "나는"은 자기 자신만, "사과를"은 둘만, "좋아한다"는 셋 다 보는 인과적 구조가
정말로 나타나는지 눈으로 확인해 봅니다.


In [ ]:
print("\n--- 순전파 (Forward Pass) ---")
scale = np.sqrt(d_head)   # √d_head : score를 눌러주는 스케일 값

# ① Q, K, V 만들기: 같은 X를 세 가지 다른 가중치에 곱해 "질문/색인표/내용물" 역할을 분리
Q = X @ W_Q;  print(f"1. Q = X @ W_Q:        shape {Q.shape}")
K = X @ W_K;  print(f"2. K = X @ W_K:        shape {K.shape}")
V = X @ W_V;  print(f"3. V = X @ W_V:        shape {V.shape}")

# ② 어텐션 점수: Q와 K를 내적(QK^T)한 뒤 √d_head로 나눠 분산을 눌러줌
S = Q @ K.T / scale
print(f"4. S = Q @ K^T / √d:   shape {S.shape}")

# ③ 인과적 마스크: i번째 토큰이 j번째(미래, j>i) 토큰을 보지 못하게 매우 작은 값을 더함
#    np.triu(..., k=1)은 "대각선 위쪽(미래 위치)"만 -1e9로 채운 행렬을 만듦
mask = np.triu(np.full((seq_len, seq_len), -1e9), k=1)
S_masked = S + mask

# ④ softmax로 확률 분포로 변환 (행 합 = 1). 마스크된 위치는 거의 0이 됨
A = softmax(S_masked)
print(f"5. A = softmax(S):     shape {A.shape}")

# ⑤ 확률(A)을 가중치로 V들을 섞어서 가져오기
O = A @ V
print(f"6. O = A @ V:          shape {O.shape}")

# ⑥ 다시 d_model 차원으로 projection
Y = O @ W_O
print(f"7. Y = O @ W_O:        shape {Y.shape}")

# ⑦ 단순한 손실: L = ||Y||^2 / 2  (워밍업 1, 2와 같은 패턴 → dL/dY = Y)
L = np.sum(Y ** 2) / 2
print(f"\n   Loss = ||Y||^2 / 2 = {L:.6f}")

# 어텐션 가중치 행렬 A를 토큰 라벨과 함께 보기 좋게 출력
print("\n어텐션 가중치 A (행 = '누가 보는가', 열 = '무엇을 보는가'):")
header = "          " + "".join(f"{t:>10s}" for t in tokens)
print(header)
for i, tok in enumerate(tokens):
    row = "".join(f"{A[i, j]:10.4f}" for j in range(seq_len))
    print(f"{tok:>10s}{row}")
print("\n(인과적 마스크 덕분에, '나는'은 자기 자신만 / '사과를'은 처음 둘만 / '좋아한다'는 셋 다 보는")
print(" 구조가 위 표에서 0.0000 으로 가려진 칸들로 그대로 나타납니다.)")

## 3단계 — 역전파(Backward Pass): 손실에서부터 거꾸로

이제 2단계의 "지도"를 **오른쪽 끝(손실 $L$)에서 시작해서 왼쪽(가중치들)으로 거슬러 올라갑니다.**
순전파가 ①→⑦ 순서였다면, 역전파는 ⑦→①의 정반대 순서로 진행됩니다. 매 단계에서 하는 일은 항상 같습니다.

> "방금 구한 *출력 쪽* 그래디언트"를 가지고, 워밍업 ①(체인룰) 또는 워밍업 ②(행렬곱 황금 규칙)를
> 적용해서 "이번 단계의 *입력 쪽* 그래디언트"를 구한다 — 이것을 계산 그래프의 끝까지 반복한다.

진행 순서를 미리 적어두면 다음과 같습니다 (괄호 안은 사용할 워밍업 규칙):

```
dL/dY                                         (시작점: L = ||Y||^2/2 이므로 dL/dY = Y)
  → dL/dW_O, dL/dO        (Y = O W_O, 황금 규칙)
  → dL/dA, dL/dV          (O = A V,   황금 규칙)
  → dL/dS                 (A = softmax(S), 이번 단계의 신규 규칙 — 가장 까다로운 부분)
  → dL/dQ, dL/dK          (S = Q K^T / √d, 황금 규칙 + 스케일)
  → dL/dW_Q, dL/dW_K, dL/dW_V   (Q=XW_Q 등, 황금 규칙)
  → (보너스) dL/dX         (X는 세 경로로 동시에 쓰였으므로 세 그래디언트를 더함)
```

표에 있는 모든 화살표가 "황금 규칙 1, 2번"의 반복일 뿐이라는 것을 알면, 어텐션 역전파가 갑자기
훨씬 덜 무섭게 느껴질 것입니다. 유일하게 새로 배워야 할 것은 **softmax의 역전파** 하나뿐입니다
(3-3에서 별도로 충분히 연습합니다). 하나씩 차례로 가보겠습니다.


### 3-1. 손실에서 출발: $dL/dY$, $dL/dW_O$, $dL/dO$

**시작점 $dL/dY$**: $L = \frac{1}{2}\sum Y^2$ 이므로, 워밍업 ①에서 본 $\frac{d}{dx}\left(\frac{x^2}{2}\right)=x$ 와
똑같은 원리로 $\frac{\partial L}{\partial Y} = Y$ 입니다. (행렬의 각 원소별로 적용되는 것뿐, 원리는 스칼라와 동일합니다.)

**$Y = OW_O$ 에 황금 규칙 적용**: 이 식은 워밍업 ②의 $Y=AB$ 꼴과 똑같습니다 ($A \to O$, $B \to W_O$).
그러므로 그대로 규칙을 가져다 쓰면 됩니다.

$$ \frac{\partial L}{\partial W_O} = O^\top \, dY \qquad\qquad \frac{\partial L}{\partial O} = dY \, W_O^\top $$


In [ ]:
print("\n--- 역전파 (Backward Pass) ---")

# 시작점: L = ||Y||^2 / 2  →  dL/dY = Y  (워밍업 1과 동일한 원리)
dL_dY = Y.copy()
print(f"7. dL/dY = Y:              shape {dL_dY.shape}")

# Y = O @ W_O 에 황금 규칙 적용 (A=O, B=W_O 로 생각)
dL_dW_O = O.T @ dL_dY          # 규칙 2: dB = A^T @ dY
print(f"   dL/dW_O = O^T @ dL/dY:  shape {dL_dW_O.shape}  (W_O와 같은 shape이어야 함: {W_O.shape})")

dL_dO = dL_dY @ W_O.T          # 규칙 1: dA = dY @ B^T
print(f"6. dL/dO = dL/dY @ W_O^T: shape {dL_dO.shape}  (O와 같은 shape이어야 함: {O.shape})")

### 3-2. 어텐션 가중치까지: $dL/dA$, $dL/dV$

다음 단계는 $O = AV$ 입니다. 이 역시 워밍업 ②와 똑같은 $Y=AB$ 형태이므로 ($A \to A$, $B \to V$),
황금 규칙을 그대로 적용합니다.

$$ \frac{\partial L}{\partial A} = dO \, V^\top \qquad\qquad \frac{\partial L}{\partial V} = A^\top \, dO $$

여기서부터는 두 가지 변수 이름이 둘 다 $A$라서 살짝 헷갈릴 수 있는데, 워밍업의 $A$(임의의 행렬)와
실제 어텐션의 $A$(softmax 결과인 어텐션 가중치)는 우연히 같은 글자를 쓴 것일 뿐, 역할은 똑같습니다 —
"$O=AV$ 식에서 왼쪽 인자에 해당하는 행렬"이라는 자리만 보고 규칙을 적용하면 됩니다.


In [ ]:
# O = A @ V 에 황금 규칙 적용 (A=A, B=V 로 생각)
dL_dA = dL_dO @ V.T            # 규칙 1: dA = dY @ B^T
print(f"   dL/dA = dL/dO @ V^T:    shape {dL_dA.shape}  (A와 같은 shape이어야 함: {A.shape})")

dL_dV = A.T @ dL_dO             # 규칙 2: dB = A^T @ dY
print(f"   dL/dV = A^T @ dL/dO:    shape {dL_dV.shape}  (V와 같은 shape이어야 함: {V.shape})")

### 3-3. 소프트맥스 역전파: 가장 까다로운 부분

지금까지는 전부 "행렬곱 황금 규칙"의 반복이었습니다. 이제 어텐션에만 등장하는 새로운 연산,
**softmax의 역전파**를 다룹니다. 다른 부분보다 조금 더 길지만, 작은 예제로 천천히 따라가면 충분히 이해할 수 있습니다.

**softmax 자체를 다시 확인**: 길이 $n$인 벡터 $s = (s_1, ..., s_n)$ 에 대해

$$ a_i = \text{softmax}(s)_i = \frac{e^{s_i}}{\sum_{k=1}^n e^{s_k}} $$

핵심 어려움은, **$a_i$가 $s_j$ 단 하나에만 의존하는 게 아니라 분모 때문에 $s$ 전체에 의존**한다는 점입니다.
($s_j$가 바뀌면 분모 $\sum_k e^{s_k}$도 바뀌고, 그 결과 $a_i$도 영향을 받습니다 — $i\ne j$ 라도요.)
그래서 $\partial a_i/\partial s_j$ 를 모든 $(i,j)$ 쌍에 대해 구한 **야코비안(Jacobian) 행렬**이 필요합니다.

**$\partial a_i / \partial s_j$ 유도** (몫의 미분법을 사용합니다)

경우 1) $i = j$ 인 경우:

$$ \frac{\partial a_i}{\partial s_i} = a_i (1 - a_i) $$

경우 2) $i \ne j$ 인 경우:

$$ \frac{\partial a_i}{\partial s_j} = -a_i a_j $$

두 경우를 하나의 식으로 합치면 (이때 $\delta_{ij}$는 "$i=j$면 1, 아니면 0"인 **크로네커 델타**입니다):

$$ \frac{\partial a_i}{\partial s_j} = a_i(\delta_{ij} - a_j) $$

**체인룰로 $dL/ds_j$ 구하기**: 어텐션 가중치 $a$ 각각이 손실에 미치는 영향 $dL/da_i$ 를 이미 알고 있다고
하면 (이전 단계에서 넘어온 $dL/dA$), 다변수 체인룰(여러 경로의 영향을 모두 더하는 규칙)에 따라

$$ \frac{\partial L}{\partial s_j} = \sum_{i=1}^n \frac{\partial L}{\partial a_i}\frac{\partial a_i}{\partial s_j}
 = \sum_{i=1}^n \frac{\partial L}{\partial a_i}\, a_i(\delta_{ij}-a_j) $$

이 합을 $\delta_{ij}$가 1인 항($i=j$)과 나머지로 나누어 정리하면:

$$ \frac{\partial L}{\partial s_j} = a_j\frac{\partial L}{\partial a_j} - a_j \sum_{i=1}^n \frac{\partial L}{\partial a_i} a_i
 = a_j\left(\frac{\partial L}{\partial a_j} - \sum_{i=1}^n \frac{\partial L}{\partial a_i}a_i\right) $$

이것이 바로 코드에서 쓸 벡터화된 공식입니다. 말로 풀면: **"내 그래디언트에서, 전체 가중 평균 그래디언트를
뺀 만큼만 반영한다"** — softmax는 "합이 1"이라는 제약이 있기 때문에, 한 항목이 커지면 다른 항목들은
자동으로 줄어들어야 하고, 이 식의 뒤쪽 항(평균을 빼주는 부분)이 바로 그 제약을 반영하는 부분입니다.

코드에서 직접 만들어보고, 같은 결과가 나오는지 (1) 이중 for문으로 만든 야코비안을 직접 곱한 값,
(2) 위에서 유도한 벡터화 공식, (3) 수치 미분 — 이렇게 세 가지 방법으로 서로 비교해서 확인해 봅니다.


In [ ]:
# 작은 예제로 softmax 역전파 공식을 직접 검증하기 (실제 어텐션과는 별개의, 길이 3짜리 toy 벡터)
s_demo = np.array([0.5, -1.2, 2.0])
a_demo = softmax(s_demo)
print("s_demo:", s_demo)
print("a_demo = softmax(s_demo):", a_demo, " (합 =", a_demo.sum(), ")")

n = len(s_demo)

# (1) 야코비안 행렬을 정의 그대로 이중 for문으로 직접 만들기: J[i,j] = d a_i / d s_j
J = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        delta_ij = 1.0 if i == j else 0.0
        J[i, j] = a_demo[i] * (delta_ij - a_demo[j])
print("\n야코비안 J = d(softmax)_i / d(s)_j:\n", J)

# 임의의 업스트림 그래디언트 dL/da 를 하나 만들어서 비교 (실제로는 이전 단계에서 넘어오는 값)
rng2 = np.random.default_rng(7)
dL_da_demo = rng2.normal(size=n)

# (1) 정의대로: dL/ds_j = sum_i dL/da_i * J[i,j]  ->  J^T @ dL_da
dL_ds_via_jacobian = J.T @ dL_da_demo

# (2) 벡터화 공식: dL/ds_j = a_j * (dL/da_j - sum_i dL/da_i * a_i)
dL_ds_via_formula = a_demo * (dL_da_demo - np.sum(dL_da_demo * a_demo))

print("\n(1) 야코비안 직접 곱     :", dL_ds_via_jacobian)
print("(2) 벡터화 공식           :", dL_ds_via_formula)
print("    -> 두 값의 최대 차이  :", np.max(np.abs(dL_ds_via_jacobian - dL_ds_via_formula)))

# (3) 수치 미분으로 한 번 더 확인. "loss"를 dL_da_demo와 softmax(s)의 내적으로 정의하면,
#     이 loss를 s로 미분한 값이 정확히 위에서 구한 dL/ds 와 같아집니다.
def toy_softmax_loss(s_):
    return np.dot(dL_da_demo, softmax(s_))

eps = 1e-6
dL_ds_numeric = np.zeros(n)
for j in range(n):
    sp = s_demo.copy(); sp[j] += eps
    sm = s_demo.copy(); sm[j] -= eps
    dL_ds_numeric[j] = (toy_softmax_loss(sp) - toy_softmax_loss(sm)) / (2 * eps)

print("(3) 수치 미분             :", dL_ds_numeric)
print("\n세 가지 방법이 모두 거의 같다면, 벡터화 공식을 믿고 실제 어텐션 행렬에 적용해도 좋습니다.")

**실제 어텐션 행렬에 적용하기**: 우리의 $A$는 벡터가 아니라 $(\text{seq\_len}, \text{seq\_len})$
모양의 **행렬**이지만, softmax는 **각 행(토큰)마다 독립적으로** 적용된다는 점을 기억하면 됩니다
(토큰 하나의 어텐션 확률은 그 토큰 행에서만 합이 1이 되고, 다른 행과는 무관합니다). 그래서 방금 만든
$n=3$짜리 벡터 공식을, $A$의 각 행에 대해 동시에 적용하면 됩니다 — `axis=-1` 방향으로 합을 구하면
모든 행에 대해 한꺼번에 계산할 수 있습니다.

**마스킹된 위치는 그래디언트도 0으로**: 마스크로 가려진 위치(미래 토큰)는 순전파에서 확률이 거의 0이라
출력에 어떤 영향도 주지 않았습니다. 이론적으로는 그래디언트도 자연히 0에 가깝게 나오지만,
$-10^9$는 진짜 $-\infty$가 아니라서 부동소수점 연산 중 아주 작은 수치 오차가 남을 수 있습니다.
그런 오차가 쌓여 학습에 영향을 주는 것을 막기 위해, **마스킹된 위치의 그래디언트는 명시적으로 0으로
만들어 줍니다** (어차피 그 위치가 의미하는 "미래를 보면 안 된다"는 규칙은 그래디언트에서도 지켜져야 하니까요).


In [ ]:
# 5. 소프트맥스 역전파를 실제 어텐션 행렬 A (shape: seq_len x seq_len) 에 적용
#    각 행(axis=-1)마다 독립적으로 3-3에서 유도한 공식을 적용합니다.
#    dL/dS[i,j] = A[i,j] * (dL/dA[i,j] - sum_k dL/dA[i,k] * A[i,k])
dL_dS = A * (dL_dA - np.sum(dL_dA * A, axis=-1, keepdims=True))
print(f"5. dL/dS (softmax grad, 마스킹 전): shape {dL_dS.shape}")

# 마스킹된 위치(미래 토큰, mask가 -1e9였던 칸)의 그래디언트는 명시적으로 0으로 만들어 줍니다.
dL_dS = np.where(mask == 0, dL_dS, 0)
print(f"   마스킹 후 dL/dS:\n{dL_dS}")
print("   (마스킹된 칸, 즉 미래를 보던 칸들이 정확히 0이 되었는지 확인해 보세요.)")

### 3-4. 점수 행렬에서 $Q$, $K$까지: $dL/dQ$, $dL/dK$

$S = QK^\top / \sqrt{d_{head}}$ 에는 두 가지를 같이 처리해야 합니다: **행렬곱**과 **상수로 나누기**.

**상수로 나누는 부분 먼저**: $\sqrt{d_{head}}$는 그냥 숫자(상수)이므로, 체인룰에 의해
"나누기"의 그래디언트는 단순히 같은 상수로 한 번 더 나누면 됩니다 — 워밍업 1에서 $g(u)=3u+5$의
미분이 항상 3이었던 것과 같은 원리로, 상수를 곱하거나 나누는 연산의 미분은 그 상수 자체입니다.

**행렬곱 부분 — 주의할 점**: $S = Q (K^\top)$ 형태이므로, 황금 규칙에서 "$B$" 자리에 들어가는 것은
$K$가 아니라 **$K^\top$**입니다. 규칙 1을 그대로 쓰면 $dQ = dS \cdot (K^\top)^\top = dS \cdot K$가 되어,
$K$를 다시 전치(transpose)하지 않고 그냥 곱하게 됩니다 — 처음 볼 때 헷갈리기 쉬운 부분이니 주의하세요.
$K$ 쪽은 "$A$" 자리에 $Q$가, "$B$" 자리에 $K^\top$가 들어간 것으로 보고 규칙 2를 적용하면
$dK^\top = Q^\top dS$, 즉 $dK = (Q^\top dS)^\top = dS^\top Q$ 가 됩니다.

$$ \frac{\partial L}{\partial Q} = \left(\frac{\partial L}{\partial S} \, K\right)\Big/\sqrt{d_{head}}
   \qquad\qquad
   \frac{\partial L}{\partial K} = \left(\left(\frac{\partial L}{\partial S}\right)^\top Q\right)\Big/\sqrt{d_{head}} $$


In [ ]:
# S = Q @ K.T / scale 의 역전파
# K 자리에는 K.T가 들어가 있었으므로, dQ를 구할 때는 K를 "다시" 전치하지 않고 그대로 곱합니다.
dL_dQ = dL_dS @ K / scale
print(f"4. dL/dQ = dL/dS @ K / sqrt(d):  shape {dL_dQ.shape}  (Q와 같은 shape: {Q.shape})")

# K에 대한 그래디언트는 dS를 먼저 전치한 뒤 Q를 곱합니다.
dL_dK = dL_dS.T @ Q / scale
print(f"   dL/dK = dL/dS^T @ Q / sqrt(d):  shape {dL_dK.shape}  (K와 같은 shape: {K.shape})")

### 3-5. 가중치 행렬까지: $dL/dW_Q$, $dL/dW_K$, $dL/dW_V$

마지막으로 $Q=XW_Q$, $K=XW_K$, $V=XW_V$ 에 황금 규칙(규칙 2)을 적용하면 끝입니다. 이번엔 모두
"$A$" 자리에 $X$가, "$B$" 자리에 각 가중치가 들어간 모양이라 규칙 2 ($dB = A^\top dY$) 만 사용합니다.

$$ \frac{\partial L}{\partial W_Q} = X^\top \frac{\partial L}{\partial Q}
   \qquad
   \frac{\partial L}{\partial W_K} = X^\top \frac{\partial L}{\partial K}
   \qquad
   \frac{\partial L}{\partial W_V} = X^\top \frac{\partial L}{\partial V} $$


In [ ]:
# Q = X @ W_Q, K = X @ W_K, V = X @ W_V 에 황금 규칙(규칙 2)을 적용
dL_dW_Q = X.T @ dL_dQ
dL_dW_K = X.T @ dL_dK
dL_dW_V = X.T @ dL_dV

print(f"1. dL/dW_Q:  shape {dL_dW_Q.shape}  (W_Q와 같은 shape: {W_Q.shape})")
print(f"2. dL/dW_K:  shape {dL_dW_K.shape}  (W_K와 같은 shape: {W_K.shape})")
print(f"3. dL/dW_V:  shape {dL_dW_V.shape}  (W_V와 같은 shape: {W_V.shape})")
print("\n이제 Y에서 시작해 W_Q, W_K, W_V, W_O 까지 모든 가중치의 그래디언트를 구했습니다!")

### 3-6 (보너스). 입력 $X$로 가지치는 그래디언트 합치기

지도를 다시 보면, $X$는 **세 군데**($W_Q$, $W_K$, $W_V$로 가는 길)에서 동시에 사용되었습니다.
실제 트랜스포머에서는 $X$가 이전 레이어의 출력이라서, 이 레이어를 통과한 뒤에도 $X$에 대한
그래디언트를 구해 이전 레이어로 계속 흘려보내야 합니다 (그래서 "역전파"가 여러 레이어를
관통해서 진행될 수 있는 것입니다).

**다변수 체인룰의 중요한 규칙**: 어떤 변수가 **여러 경로**를 통해 출력에 영향을 미친다면,
**각 경로에서 계산한 그래디언트를 모두 더해야** 합니다. (한 경로만 보고 끝내면 다른 경로의 영향을
빠뜨리게 됩니다.) $X$는 $Q$, $K$, $V$ 세 경로 모두에 쓰였으므로:

$$ \frac{\partial L}{\partial X} = \frac{\partial L}{\partial Q} W_Q^\top
   + \frac{\partial L}{\partial K} W_K^\top
   + \frac{\partial L}{\partial V} W_V^\top $$

(각 항은 $Q=XW_Q$ 형태에 황금 규칙 1을 적용한 결과이며, $X$로 흘러드는 세 경로를 그냥 더한 것뿐입니다.)


In [ ]:
# X는 W_Q, W_K, W_V로 가는 세 경로 모두에 사용되었으므로, 각 경로의 그래디언트를 모두 더합니다.
dL_dX = dL_dQ @ W_Q.T + dL_dK @ W_K.T + dL_dV @ W_V.T
print(f"(보너스) dL/dX:  shape {dL_dX.shape}  (X와 같은 shape: {X.shape})")
print("\ndL/dX:\n", dL_dX)
print("\n실제 트랜스포머에서는 이 dL/dX가 바로 아래(이전) 레이어로 전달되는 그래디언트입니다.")

## 4단계 — 수치 미분으로 전체 결과 검증하기

이제까지 손으로 유도한 그래디언트들이 정말 맞는지, **정답을 모른 채로도** 확인할 수 있는 방법이
수치 미분(finite difference)입니다. 워밍업 ①, ②에서 이미 한 번씩 사용해 본 방법을 이번에는
**모든 가중치 행렬의 모든 원소**에 대해 체계적으로 적용해 봅니다.

**왜 중앙 차분(central difference)을 쓰는가?** $\frac{f(x+\epsilon)-f(x-\epsilon)}{2\epsilon}$ 처럼
양쪽으로 같은 만큼 움직여서 계산하면, 한쪽으로만 움직이는 방법($\frac{f(x+\epsilon)-f(x)}{\epsilon}$)보다
오차가 훨씬 작습니다 (정확히는, 오차의 크기가 $\epsilon$에 비례하는 대신 $\epsilon^2$에 비례하게 되어
훨씬 빠르게 작아집니다).

**eps는 너무 크지도, 너무 작지도 않아야 합니다.**
- eps가 너무 크면: 함수가 그 구간에서 직선이 아니라 휘어 있기 때문에(비선형성), 근사 자체가 부정확해집니다.
- eps가 너무 작으면: 컴퓨터의 부동소수점 계산은 정밀도가 한계가 있어서, $f(x+\epsilon)$과 $f(x-\epsilon)$의
  차이가 표현 가능한 가장 작은 단위보다 작아져 버려 오히려 오차가 커집니다.
- 보통 $\epsilon \approx 10^{-5} \sim 10^{-4}$ 정도가 실용적으로 잘 맞습니다. (아래 코드에서도 `1e-5`를 사용합니다.)

**상대 오차(relative error)를 보는 이유**: 그래디언트 값 자체의 크기가 천차만별이라서, 그냥 두 값의
차이(절대 오차)만 보면 큰 값끼리는 항상 차이가 크게 보이고 작은 값끼리는 항상 작게 보입니다.
그래서 $\frac{|\text{해석적} - \text{수치적}|}{|\text{해석적}| + |\text{수치적}| + \epsilon_{\text{안전}}}$
처럼 **크기에 비례해서** 오차를 측정합니다. 일반적으로 이 상대 오차가 $10^{-5}$ 이하면 구현이 맞다고
판단해도 좋습니다 (분모에 더해준 아주 작은 $\epsilon_{\text{안전}}$ 은 그래디언트가 0에 가까울 때
분모가 0이 되어 나누기 오류가 나는 것을 막기 위한 안전장치입니다).

아래에서는 `W_Q, W_K, W_V, W_O` 와, 보너스로 구한 `X`까지 — **모든 파라미터의 모든 원소**를
하나씩 흔들어보며 검증하는 범용 함수를 만들어 한 번에 확인합니다.


In [ ]:
print("\n--- 수치적 검증 (유한 차분법, 모든 원소) ---")

def compute_loss_full(X_, W_Q_, W_K_, W_V_, W_O_):
    """순전파 전체를 다시 계산해서 손실만 반환하는 함수 (수치 미분에 사용)."""
    Q_ = X_ @ W_Q_
    K_ = X_ @ W_K_
    V_ = X_ @ W_V_
    S_ = Q_ @ K_.T / scale + mask     # mask는 위에서 만든 인과적 마스크를 그대로 재사용
    A_ = softmax(S_)
    O_ = A_ @ V_
    Y_ = O_ @ W_O_
    return np.sum(Y_ ** 2) / 2

def grad_check(param, analytic_grad, name, loss_fn, eps=1e-5):
    """param의 모든 원소를 eps만큼 흔들어보며, 해당 위치의 해석적 그래디언트가 맞는지 검증.
    모든 원소 중 가장 큰 상대 오차(max relative error)를 출력합니다."""
    max_rel_err = 0.0
    it = np.nditer(param, flags=["multi_index"])
    for _ in it:
        idx = it.multi_index
        original = param[idx]

        param[idx] = original + eps
        loss_plus = loss_fn()

        param[idx] = original - eps
        loss_minus = loss_fn()

        param[idx] = original   # 원래 값으로 복원 (다음 원소를 검사하기 전에 꼭 복원!)

        numerical = (loss_plus - loss_minus) / (2 * eps)
        analytic = analytic_grad[idx]
        rel_err = abs(analytic - numerical) / (abs(analytic) + abs(numerical) + 1e-10)
        max_rel_err = max(max_rel_err, rel_err)

    status = "OK" if max_rel_err < 1e-5 else "확인 필요"
    print(f"  {name:12s}: 최대 상대 오차 = {max_rel_err:.2e}   [{status}]")
    return max_rel_err

grad_check(W_Q, dL_dW_Q, "dL/dW_Q", lambda: compute_loss_full(X, W_Q, W_K, W_V, W_O))
grad_check(W_K, dL_dW_K, "dL/dW_K", lambda: compute_loss_full(X, W_Q, W_K, W_V, W_O))
grad_check(W_V, dL_dW_V, "dL/dW_V", lambda: compute_loss_full(X, W_Q, W_K, W_V, W_O))
grad_check(W_O, dL_dW_O, "dL/dW_O", lambda: compute_loss_full(X, W_Q, W_K, W_V, W_O))
grad_check(X,   dL_dX,   "dL/dX (보너스)", lambda: compute_loss_full(X, W_Q, W_K, W_V, W_O))

print("""
요약:
  - 어텐션 역전파는 순전파의 역순으로, 워밍업에서 연습한 두 가지 황금 규칙을 반복 적용한 것뿐입니다.
  - 유일한 신규 규칙은 softmax 역전파(야코비안)였고, 작은 예제로 충분히 연습한 뒤 그대로 적용했습니다.
  - 수치 미분으로 모든 원소를 검증해, 우리가 손으로 유도한 공식이 실제로 맞다는 것을 확인했습니다.
  - 실제 PyTorch/JAX 같은 프레임워크는 바로 이 과정을 자동으로(autograd) 수행해 줍니다!
""")

## 5단계 — 실험해 보기: 이것저것 바꿔보며 감 잡기

지금까지는 이론을 코드로 옮기는 데 집중했습니다. 이제 값을 바꿔보면서 **"무엇이 무엇에 영향을 미치는가"**
를 직접 느껴봅시다. 아래 코드는 d_head, seq_len, scale 등 핵심 하이퍼파라미터를 바꿔가며
어텐션 가중치(A), 그래디언트 크기 등이 어떻게 달라지는지 보여줍니다.

**실험 1 — scale(√d_head)을 빼면 어떻게 될까?**
scale 없이 점수를 계산하면 값이 매우 커져서 softmax가 한 위치에 거의 1을 몰아주고 나머지는 0에
가깝게 됩니다. 이 현상을 "softmax 포화(saturation)"라고 부르며, 포화된 softmax의 그래디언트는
거의 0이 되어 학습이 매우 느려집니다.

**실험 2 — seq_len을 늘리면?**
길이가 길어질수록 "볼 수 있는 과거 토큰"이 많아지고, 첫 번째 토큰의 어텐션은 계속 자기 자신만 보지만
마지막 토큰은 모든 토큰을 골고루 보게 됩니다 — 각 위치마다 그래디언트 분포가 달라집니다.

아래 코드에서 직접 확인해 보고, 마음에 드는 값으로 바꿔서 실행해 보세요.


In [ ]:
print("=== 실험 ① : scale을 빼면 softmax가 포화되는가? ===")

# 같은 Q, K로 두 가지 버전의 어텐션 점수를 계산
S_no_scale = Q @ K.T                  # scale 없는 버전
S_with_scale = Q @ K.T / scale        # scale 있는 버전

A_no_scale    = softmax(S_no_scale    + mask)
A_with_scale  = softmax(S_with_scale  + mask)

print("\nscale 없을 때 A (포화 여부 확인 - 한 값만 1에 가깝고 나머지가 0에 가까워지면 포화):")
for i, tok in enumerate(tokens):
    row = "".join(f"{A_no_scale[i, j]:8.4f}" for j in range(seq_len))
    print(f"  {tok:>8s}: {row}")

print("\nscale 있을 때 A (보통 더 고르게 분포):")
for i, tok in enumerate(tokens):
    row = "".join(f"{A_with_scale[i, j]:8.4f}" for j in range(seq_len))
    print(f"  {tok:>8s}: {row}")

print("\n→ scale이 없으면 점수 분포가 과하게 날카로워져, 한 토큰이 거의 모든 attention을 독점합니다.")
print("  scale이 있으면 상대적으로 고른 분포가 나와 모든 위치에서 그래디언트가 살아남습니다.\n")

print("=== 실험 ② : seq_len을 5로 늘리면? ===")
np.random.seed(0)
seq5 = 5
X5   = np.random.randn(seq5, d_model) * 0.5
Q5   = X5 @ W_Q
K5   = X5 @ W_K
S5   = Q5 @ K5.T / scale
mask5 = np.triu(np.full((seq5, seq5), -1e9), k=1)
A5   = softmax(S5 + mask5)

tok5 = [f"T{i}" for i in range(seq5)]
print("seq_len=5일 때 어텐션 가중치 A5 (행=누가보나, 열=무엇을보나):")
header = "       " + "".join(f"{t:>7s}" for t in tok5)
print(header)
for i, tok in enumerate(tok5):
    row = "".join(f"{A5[i, j]:7.3f}" for j in range(seq5))
    print(f"{tok:>6s}: {row}")
print("\n→ T0은 항상 자기만 봄(1.000), T4는 다섯 개를 모두 어느 정도 봄을 확인하세요.")
print("  인과적 구조는 길이가 달라져도 항상 유지됩니다.")

## 6단계 — 역전파 전체 치트시트

아래 코드 셀은 이 노트북에서 배운 역전파의 모든 단계를 한눈에 볼 수 있는 다이어그램을 출력합니다.
나중에 어텐션 역전파가 헷갈릴 때 이 출력을 참고하세요.


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║           어텐션 역전파 전체 치트시트 (numpy 기준)                ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  [순전파]                         [역전파 — 역순으로 적용]        ║
║  ─────────────────────────────    ──────────────────────────────  ║
║                                                                  ║
║  X──┬─W_Q──▶Q─┐                  dL/dW_Q = X.T @ dL/dQ         ║
║     ├─W_K──▶K─┤──▶ S=QKᵀ/√d     dL/dW_K = X.T @ dL/dK         ║
║     └─W_V──▶V─┘                  dL/dW_V = X.T @ dL/dV         ║
║                 │                                                ║
║                 ▼   (+mask)       dL/dQ = dL/dS @ K / √d        ║
║              S_masked             dL/dK = dL/dSᵀ @ Q / √d       ║
║                 │                                                ║
║                 ▼                 ★ softmax 역전파 (핵심):        ║
║          A=softmax(S) ──────────▶ dL/dS = A*(dL/dA − Σ(dL/dA*A))║
║                 │                 (마스킹된 위치는 0으로 덮어씀)   ║
║                 ▼                                                ║
║           O = A @ V ────────────▶ dL/dA = dL/dO @ Vᵀ            ║
║                 │                 dL/dV = Aᵀ @ dL/dO             ║
║                 ▼                                                ║
║           Y = O @ W_O ──────────▶ dL/dW_O = Oᵀ @ dL/dY          ║
║                 │                 dL/dO   = dL/dY @ W_Oᵀ         ║
║                 ▼                                                ║
║           L = ½‖Y‖² ───────────▶ dL/dY = Y   (출발점)            ║
║                                                                  ║
║  (보너스) dL/dX = dL/dQ@W_Qᵀ + dL/dK@W_Kᵀ + dL/dV@W_Vᵀ         ║
║           (X가 세 경로에 쓰였으므로 세 그래디언트를 모두 더함)      ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║  황금 규칙 요약 (Y = A @ B 형태의 행렬곱)                         ║
║    규칙 1: dA = dY @ Bᵀ    규칙 2: dB = Aᵀ @ dY                 ║
╚══════════════════════════════════════════════════════════════════╝
""")

## 마무리 — 오늘 배운 것 정리

이 노트북에서 배운 내용을 한 문장씩 정리합니다.

**체인룰(연쇄법칙)**
복잡한 합성함수의 미분은 각 단계의 미분을 순서대로 "곱"하면 됩니다.
신경망의 역전파는 이 곱을 계산 그래프 끝에서 처음까지 체계적으로 적용한 것이 전부입니다.

**행렬곱의 황금 규칙 두 가지**
$Y = AB$ 꼴의 행렬곱에서, $dA = dY \cdot B^\top$ 이고 $dB = A^\top \cdot dY$ 입니다.
어텐션 역전파의 거의 모든 단계는 이 두 규칙의 반복 적용으로 해결됩니다.

**softmax 역전파**
한 원소가 나머지 전체에 영향을 주는(합이 1이 되어야 하는) 구조 때문에, 야코비안(편미분 행렬)이
필요합니다. 최종 공식 $dS = A \odot (dA - \text{row\_sum}(dA \odot A))$ 으로 벡터화할 수 있습니다.

**인과적 마스크(Causal mask)**
순전파에서 미래 위치를 $-10^9$로 가렸던 것처럼, 역전파에서도 그 위치의 그래디언트는 0으로
명시적으로 만들어 줍니다 — "앞으로 보지 않는다"는 규칙은 순전파뿐 아니라 역전파에서도 지켜져야
합니다.

**다중 경로 그래디언트 합산**
$X$처럼 여러 경로에 동시에 사용된 변수는, 각 경로에서 따로 계산한 그래디언트를 모두 더합니다.

**수치 미분(finite difference) 으로 검증**
$(f(x+\epsilon) - f(x-\epsilon)) / (2\epsilon)$ 을 이용하면, 우리가 손으로 유도한 공식이
"정답 없이도" 맞는지 검증할 수 있습니다. 구현 중 실수가 있는지 확인하는 가장 실용적인 도구입니다.

**실제 프레임워크는 이걸 자동으로**
PyTorch, JAX 같은 딥러닝 프레임워크의 `autograd`(자동 미분) 기능은 바로 이 과정을 — 계산 그래프를
기록하고, 역방향으로 체인룰을 적용하고, 각 파라미터에 그래디언트를 쌓아두는 일을 — 자동으로
해줍니다. 이 노트북에서 직접 손으로 해본 덕분에, 이제는 `loss.backward()` 한 줄이 내부적으로
무슨 일을 하는지 알게 되었습니다.

---

*추가로 공부하면 좋은 것들*: 다중 헤드 어텐션(Multi-Head Attention)에서 헤드가 여러 개일 때
그래디언트가 어떻게 합쳐지는지, LayerNorm의 역전파, Flash Attention에서 메모리를 줄이면서
역전파를 어떻게 수행하는지 등을 차례로 살펴보면 트랜스포머 전체를 바닥부터 이해하게 됩니다.
